# 03 — 直接模式：恒电位仪控制

**直接模式**允许你手动控制恒电位仪 — 即时设置电位或电流并读取测量值 — 无需加载方法文件。

直接模式适用于：
- 快速点测量（OCV、保持电位）
- 条件处理或平衡步骤
- 设备或电池设置的故障排查
- 用 Python 构建自定义测量循环

自动化实验序列（LSV、CV、EIS 等）请参阅 **`06_method_mode.ipynb`**。

### 前提条件
- IviumSoft 正在运行
- 设备已连接（参见 `02_device_and_instance_management.ipynb`）
- 错误处理参考：`01_getting_started.ipynb` — 第 4 节

In [ ]:
import time
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
print("就绪")

## 1. 连接模式

连接模式决定电极接线方式。打开电池前务必先设置。

| 代码 | 模式 | 使用场景 |
|------|------|----------|
| 0 | Off | 无连接 |
| 1 | EStat 4EL | 标准 4 电极恒电位仪（默认） |
| 2 | EStat 2EL | 2 电极恒电位仪 |
| 3–6 | EStat Dummy | 虚拟电池（测试/校准） |
| 7 | IStat 4EL | 4 电极恒电流仪 |
| 8 | IStat 2EL | 2 电极恒电流仪 |
| 9 | IStat Dummy | 恒电流仪虚拟电池 |
| 10 | BiStat 4EL | 双恒电位仪 4 电极 |
| 11 | BiStat 2EL | 双恒电位仪 2 电极 |

In [ ]:
# 设置为标准 4 电极恒电位仪
Pyvium.set_connection_mode(1)
print("连接模式：EStat 4EL（标准恒电位仪）")

## 2. 电池状态

`get_cell_status()` 返回从位掩码解码出的活跃状态标志列表。各标志及对应位：

| 标志 | 位 | 含义 |
|------|-----|------|
| `'Cell on'` | — | 电池输出已激活 |
| `'Cell off'` | — | 电池输出已禁用（连接后的默认状态） |
| `'I_ovl'` | 2 | 电流过载 — 请降低电流量程 |
| `'Anin1_ovl'` | 4 | 模拟输入 1 过载 |
| `'E_ovl'` | 5 | 电位过载 — 请降低电位量程 |
| `'CellOff_button pressed'` | 7 | 硬件安全按钮被按下 |

当 `'I_ovl'` 或 `'E_ovl'` 激活时，测量结果不可靠 — 请检查电流量程和施加电位。

> **重要：** 通过 `start_method()` 运行方法期间，直接模式设置（电位、电流量程、滤波器、稳定性）会**暂停**。方法完成或中止后恢复。仅在无方法运行时配置直接模式设置。

In [ ]:
status = Pyvium.get_cell_status()
print(f"电池状态标志: {status}")

# 读取测量值前先检查过载
OVERLOAD_FLAGS = {'I_ovl', 'E_ovl', 'Anin1_ovl'}
overloads = OVERLOAD_FLAGS & set(status)
if overloads:
    print(f"警告：检测到过载: {overloads}")
else:
    print("无过载 — 测量结果可信")

## 3. 电流量程

打开电池**前**先设置电流量程。量程过大会降低分辨率；量程过小会导致过载。

DLL 将电流量程代码定义为降十进制序列。根据 IviumSoft DLL 参考手册，`IV_setcurrentrange` 从 **0 = 10 A、1 = 1 A** 开始，按十进制依次递减：

| 代码 | 量程（IviumStat / 全量程设备） |
|------|--------------------------------|
| 0 | 10 A |
| 1 | 1 A |
| 2 | 100 mA |
| 3 | 10 mA |
| 4 | 1 mA |
| 5 | 100 µA |
| 6 | 10 µA |
| 7 | 1 µA |
| 8 | 100 nA |
| … | … |

小电流设备（CompactStat、pocketSTAT）仅支持较低量程的子集 — 代码仍遵循相同的降十进制规律，但设备的最大量程低于 10 A，因此代码 0 对应该型号最高可用量程。

> **在脚本中使用这些值前，务必通过 IviumSoft 直接模式电流量程下拉列表确认你设备的正确代码。**

> 对于 WE2（BiStat），刻度不同：`IV_setcurrentrangeWE2` 从 0 = 10 mA、1 = 1 mA 等开始。参见笔记本 `05_bipotentiostat_and_we32.ipynb`。

In [ ]:
Pyvium.set_current_range(4)  # 1 mA 量程
print("电流量程设置为 1 mA")

## 4. 滤波器和稳定性设置

**滤波器**控制电流测量的带宽。带宽越低 = 噪声越小，但响应越慢。

| 代码 | 滤波器 |
|------|--------|
| 0 | 1 MHz（最快，噪声最大） |
| 1 | 100 kHz |
| 2 | 10 kHz |
| 3 | 1 kHz |
| 4 | 10 Hz（最慢，噪声最小） |

**稳定性**根据电池阻抗调整恒电位仪反馈回路：

| 代码 | 稳定性 | 适用场景 |
|------|--------|----------|
| 0 | 高速 | 低阻抗电池、快速扫描 |
| 1 | 标准 | 大多数电化学电池（默认） |
| 2 | 高稳定性 | 高阻抗电池、噪声信号 |

In [ ]:
Pyvium.set_filter(2)       # 10 kHz 滤波器
Pyvium.set_stability(1)    # 标准稳定性
print("滤波器: 10 kHz | 稳定性: 标准")

## 5. 打开电池并设置电位

> **安全提示：** 始终在打开电池**之前**设置目标电位，以防止出现意外值的瞬态尖峰。

In [ ]:
# 打开电池前先设置电位
Pyvium.set_potential(0.1)  # 相对参比的 0 V
print("目标电位: 0.1 V")

Pyvium.set_cell_on()
status = Pyvium.get_cell_status()
print(f"电池状态: {status}")
print(f"电位: {Pyvium.get_potential()}")

## 6. 读取电位和电流

电池打开后读取电位和电流。改变电位后需给恒电位仪短暂的稳定时间。

In [ ]:
time.sleep(0.1)  # 等待稳定

potential = Pyvium.get_potential()
current   = Pyvium.get_current()

print(f"测量电位 : {potential:.6f} V")
print(f"测量电流 : {current:.3e} A")

## 7. 恒电位保持 — 随时间采样

常用模式：保持电位固定并记录电流随时间的变化（计时安培法）。

In [ ]:
HOLD_POTENTIAL = 0.2   # V
DURATION_S     = 5.0   # 秒
INTERVAL_S     = 0.2   # 采样间隔

Pyvium.set_potential(HOLD_POTENTIAL)
Pyvium.set_cell_on()
time.sleep(0.1)  # 稳定

timestamps = []
currents   = []
t_start    = time.time()

while (time.time() - t_start) < DURATION_S:
    t = time.time() - t_start
    I = Pyvium.get_current()
    timestamps.append(t)
    currents.append(I * 1e6)  # 转换为 µA
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()
print(f"在 {DURATION_S:.0f} s 内采集了 {len(timestamps)} 个点")

# 绘图
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(timestamps, currents, 'b-o', markersize=3)
ax.set_xlabel("时间 (s)")
ax.set_ylabel("电流 (µA)")
ax.set_title(f"计时安培法 — 保持在 {HOLD_POTENTIAL} V")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 恒电流模式（设置电流）

在恒电流模式下，仪器控制电流，你读取电位。首先切换到 IStat 连接模式。

In [ ]:
HOLD_CURRENT = 1e-6  # 1 µA

Pyvium.set_connection_mode(7)   # IStat 4EL（恒电流仪）
Pyvium.set_current_range(4)     # 1 µA 量程
Pyvium.set_current(HOLD_CURRENT)
Pyvium.set_cell_on()
time.sleep(0.1)

potential = Pyvium.get_potential()
current   = Pyvium.get_current()
print(f"施加电流  : {current:.3e} A")
print(f"测量电位  : {potential:.6f} V")

Pyvium.set_cell_off()
Pyvium.set_connection_mode(1)   # 返回恒电位仪模式

## 9. 开路电位（OCV）

OCV 在电池**关闭**时测量 — 仪器仅监听，不施加任何信号。

In [ ]:
OCV_DURATION_S = 10.0
INTERVAL_S     = 0.5

Pyvium.set_cell_off()  # 真实 OCV 时电池必须关闭

timestamps = []
potentials = []
t_start    = time.time()

while (time.time() - t_start) < OCV_DURATION_S:
    t = time.time() - t_start
    E = Pyvium.get_potential()
    timestamps.append(t)
    potentials.append(E)
    time.sleep(INTERVAL_S)

print(f"结束时 OCV: {potentials[-1]:.4f} V")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(timestamps, potentials, 'g-')
ax.set_xlabel("时间 (s)")
ax.set_ylabel("OCV (V)")
ax.set_title("开路电位随时间变化")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 清理

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("完成")

---

## 小结

| 任务 | 方法 |
|------|--------|
| 设置电极接线 | `set_connection_mode(n)` |
| 检查电池状态 | `get_cell_status()` |
| 设置电流量程 | `set_current_range(n)` |
| 设置测量滤波器 | `set_filter(n)` |
| 设置稳定性 | `set_stability(n)` |
| 施加电位（恒电位仪） | `set_potential(V)` |
| 施加电流（恒电流仪） | `set_current(A)` |
| 读取电位 | `get_potential()` → float |
| 读取电流 | `get_current()` → float |
| 开启 / 关闭电池 | `set_cell_on()` / `set_cell_off()` |

## 下一步

- **`04_direct_mode_signals.ipynb`** — 高速波形、DAC/ADC、数字 I/O、AC 控制
- **`05_bipotentiostat_and_we32.ipynb`** — BiStat（WE2）和 32 通道 WE32 阵列
- **`06_method_mode.ipynb`** — 运行完整电化学方法（LSV、CV、EIS……）